# SimuInterface 测试

测试 `scripts/simu_interface.py` 中的 SimuInterface 和 MockSimuInterface 类

## 使用说明

1. **MockSimuInterface**: 无需 MuJoCo 环境，可直接运行
2. **SimuInterface**: 需要安装 MuJoCo 并配置正确的 XML 文件路径

In [ ]:
# 设置项目路径（相对路径，便于移植）
from pathlib import Path
import sys

# 项目根目录
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent if '__file__' in dir() else Path.cwd().parent.parent.parent
COLLECT_DATA_ROOT = PROJECT_ROOT / 'kortex_code' / 'collect_data'

# 添加到 Python 路径
sys.path.insert(0, str(COLLECT_DATA_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"COLLECT_DATA_ROOT: {COLLECT_DATA_ROOT}")

import numpy as np
from scripts.simu_interface import SimuInterface, MockSimuInterface

## 1. 测试 MockSimuInterface（无需仿真环境）

In [ ]:
# 创建 MockSimuInterface
mock_simu = MockSimuInterface(xml_path='test.xml')
print("MockSimuInterface 对象创建成功")

# 初始化
success = mock_simu.initialize()
print(f"初始化结果: {success}")

## 2. 测试关节状态设置和获取

In [ ]:
# 获取初始关节状态
joint_state = mock_simu.get_joint_state()
print(f"初始关节状态 (弧度): {joint_state}")
print(f"初始关节状态 (角度): {np.rad2deg(joint_state)}")

# 设置关节目标位置 (使用 set_joint_target，角度制)
target_joints_deg = np.array([10.0, 20.0, 30.0, 40.0, 50.0, 60.0])
result = mock_simu.set_joint_target(target_joints_deg)
print(f"设置关节目标结果: {result}")

# 获取设置后的关节状态
joint_state = mock_simu.get_joint_state()
print(f"设置后关节状态 (弧度): {joint_state}")

## 3. 测试夹爪控制

In [ ]:
# 获取夹爪状态
gripper_state = mock_simu.get_gripper_state()
print(f"初始夹爪状态: {gripper_state} (0=张开, 1=闭合)")

# 设置夹爪位置
result = mock_simu.set_gripper(0.5)  # 半闭合
print(f"设置夹爪结果: {result}")

gripper_state = mock_simu.get_gripper_state()
print(f"设置后夹爪状态: {gripper_state}")

# 完全闭合
mock_simu.set_gripper(1.0)
print(f"完全闭合后夹爪状态: {mock_simu.get_gripper_state()}")

## 4. 测试物体位置设置和获取

In [ ]:
# 获取物体位置
cube_pos = mock_simu.get_object_position('cube')
print(f"Cube 位置: {cube_pos}")

# 设置物体位置
new_cube_pos = np.array([0.3, 0.0, 0.1])
result = mock_simu.set_object_position('cube', new_cube_pos)
print(f"设置 cube 位置结果: {result}")

# 再次获取
cube_pos = mock_simu.get_object_position('cube')
print(f"设置后 Cube 位置: {cube_pos}")

## 5. 测试渲染功能

In [ ]:
# 单相机渲染
image = mock_simu.render()
print(f"单相机渲染图像形状: {image.shape}, dtype: {image.dtype}")

# 多相机渲染
camera_images = mock_simu.get_camera_images(['agentview', 'top'])
print(f"多相机渲染结果:")
for cam_name, img in camera_images.items():
    print(f"  {cam_name}: shape={img.shape}, dtype={img.dtype}")

## 6. 测试 step 和 disconnect 功能

In [ ]:
# 调用 step
mock_simu.step(n_steps=100)
print("step(100) 调用成功")

# 断开连接
mock_simu.disconnect()
print("disconnect() 调用成功")

## 7. 测试 SimuInterface（需要 MuJoCo 仿真环境）

**注意**: 以下测试需要安装 MuJoCo 并存在对应的 XML 场景文件。如果环境未配置，将跳过测试。

In [ ]:
# 检查 MuJoCo 是否可用
try:
    import mujoco
    MUJOCO_AVAILABLE = True
    print(f"MuJoCo 版本: {mujoco.__version__}")
except ImportError:
    MUJOCO_AVAILABLE = False
    print("MuJoCo 未安装，跳过 SimuInterface 测试")

# 检查 XML 文件是否存在
xml_path = PROJECT_ROOT / 'kortex_code' / 'kortex_simu' / 'simu' / 'env' / 'task_pick_place.xml'
print(f"XML 文件路径: {xml_path}")
print(f"XML 文件存在: {xml_path.exists()}")

In [ ]:
if MUJOCO_AVAILABLE and xml_path.exists():
    print("="*60)
    print("开始测试 SimuInterface...")
    print("="*60)
    
    try:
        # 创建 SimuInterface
        simu = SimuInterface(
            xml_path=str(xml_path),
            camera_names=['agentview', 'top'],
            use_ik=False  # 禁用 IK，简化测试
        )
        print("SimuInterface 对象创建成功")
        
        # 初始化 (不显示 viewer)
        success = simu.initialize(show_viewer=False)
        print(f"初始化结果: {success}")
        
        if success:
            # 测试关节状态
            joint_state = simu.get_joint_state()
            print(f"关节状态 (弧度): {joint_state}")
            print(f"关节状态 (角度): {np.rad2deg(joint_state)}")
            
            # 测试夹爪状态
            gripper_state = simu.get_gripper_state()
            print(f"夹爪状态: {gripper_state}")
            
            # 测试设置关节目标
            target_deg = np.array([0.0, 30.0, 0.0, 60.0, 0.0, 0.0])
            result = simu.set_joint_target(target_deg)
            print(f"设置关节目标结果: {result}")
            
            # 执行仿真步
            simu.step(n_steps=100)
            print("step(100) 完成")
            
            # 测试渲染
            images = simu.get_camera_images(['agentview'])
            if 'agentview' in images:
                print(f"agentview 图像形状: {images['agentview'].shape}")
            
            # 测试物体位置
            cube_pos = simu.get_object_position('cube')
            print(f"Cube 位置: {cube_pos}")
            
            # 断开连接
            simu.disconnect()
            print("SimuInterface disconnect() 完成")
            print("\n" + "="*60)
            print("SimuInterface 测试通过!")
            print("="*60)
        else:
            print("SimuInterface 初始化失败")
            
    except Exception as e:
        print(f"SimuInterface 测试出错: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n跳过 SimuInterface 测试（MuJoCo 未安装或 XML 文件不存在）")

## 8. 接口对照表

| 功能 | SimuInterface | MockSimuInterface |
|------|---------------|-------------------|
| 初始化 | `initialize(xml_path, show_viewer)` | `initialize()` |
| 获取关节状态 | `get_joint_state()` → np.ndarray | `get_joint_state()` |
| 设置关节目标 | `set_joint_target(positions_deg)` | `set_joint_target(positions)` |
| 获取夹爪状态 | `get_gripper_state()` → float | `get_gripper_state()` |
| 设置夹爪 | `set_gripper(position)` | `set_gripper(position)` |
| 获取物体位置 | `get_object_position(name)` | `get_object_position(name)` |
| 设置物体位置 | `set_object_position(name, pos)` | `set_object_position(name, pos)` |
| 仿真步进 | `step(n_steps)` | `step(n_steps)` |
| 渲染图像 | `render(camera_name)` | `render(camera_name)` |
| 多相机渲染 | `get_camera_images(names)` | `get_camera_images(names)` |
| 断开连接 | `disconnect()` | `disconnect()` |
| IK 控制 | `move_to_cartesian(pos, ori)` | 不支持 |